# NFL Data — Colab Runner

Thin execution shell. All logic lives in the `nfl_data` package in this repo — edit that in Claude Code, not here.

This notebook: clones the repo fresh → installs it → authenticates to GCP → runs the pipeline → writes to BigQuery.

Writes to three BigQuery datasets in `ff-python-api`:
- `nflreadpy` — raw nflreadpy pulls (players, player_stats, snap_counts, nextgen_stats, ff_opportunity), replaced each run
- `dynasty` — `yprr_proxy`, a derived route-share proxy table
- `dynasty_tycoon` — `player_auction_values`, the Sleeper-projection-based dynasty auction valuation

## 1. Clone the repo

Set `REPO_URL` once. If the repo is private, add a `GITHUB_TOKEN` secret in Colab (key icon, left sidebar) first.

In [ ]:
REPO_URL = "https://github.com/nashstallings/fantasy_football.git"  # update this
REPO_DIR = "fantasy_football"

import os

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}

!git clone {clone_url} {REPO_DIR}

## 2. Install the package

In [ ]:
%cd {REPO_DIR}
!pip install -q -e .
%cd ..

## 3. Auth

GCP auth via Colab's built-in account.

In [ ]:
from google.colab import auth
auth.authenticate_user()

## 4. Run the pipeline

In [ ]:
import sys
sys.path.insert(0, f"/content/{REPO_DIR}/src")

import nfl_data

tables = nfl_data.run(write_to_bq=True)
for name, df in tables.items():
    print(name, df.shape)

## 5. Verify

Row counts per dataset (the pipeline writes to all three; check whichever you care about).

In [ ]:
from nfl_data.bigquery_io import verify
from nfl_data import config

for dataset_id in (config.NFLREADPY_DATASET_ID, config.YPRR_DATASET_ID, config.VALUATION_DATASET_ID):
    print(f"--- {dataset_id} ---")
    print(verify(config.PROJECT_ID, dataset_id))

## 6. Run a single stage (optional)

Each stage can be run independently — useful if you only need to refresh one dataset, or want to iterate on `season`/`seasons` without re-running everything.

In [ ]:
# from nfl_data import run_nflreadpy_tables, run_yprr, run_auction_values

# run_nflreadpy_tables(season=2025)
# run_yprr(seasons=list(range(2012, 2026)))
# run_auction_values(season=2026)